In [1]:
from importlib.machinery import SourceFileLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from Tools import CaseNamer, Plotting

In [2]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
### feature selection
from sklearn.feature_selection import SelectKBest, SequentialFeatureSelector
from sklearn.pipeline import Pipeline

In [3]:
%run GetPymatgenFeatures.py

StrToComposition:   0%|          | 0/1687 [00:00<?, ?it/s]

In [4]:
CASE = 'INITIAL'  #, 'INITIAL', RELAXED
MODEL = 'ATOMIC' #, 'ORTHOGONAL', 'CANONICAL'
CUTOFF = '' # 'TABLECUTOFF', 'HISTCUTOFF'
criterion = 'test_score'  # test_score, train_score, err
TARGET = 'EF'


In [5]:
# from SourceDevelopementVersion import Featurizer, FeatureConcatenate
%run SourceDevelopementVersion.py

In [6]:
BS = pd.read_pickle('parsedbs.pkl')
Features = Featurizer(BS)
groundstates = Features.get_ground_states_energies()
BS[TARGET] = Features.get_formation_energy(groundstates)

In [7]:
BS = BS[(BS[TARGET] > -1) & (BS[TARGET]< 2)]
BS = BS[(BS['E0']>-100) & (BS['E0']<10)]
BS = BS[(BS['B0'])>0 & (BS['B0']<500)]
BS.dropna(how='any', axis=0,inplace=True)

In [8]:
bestfeats = {}
bestscores = {}
FC = {}

In [9]:
CASE = 'initial'
criterion = 'test_score'

In [10]:
from Tools import CaseNamer, Plotting

In [11]:
FileNames = CaseNamer(CASE, 'AtomicFeatures', '', '', criterion, '',)

# Atomic Features 

In [12]:
AtomicFeaturesMagpie['MagConfig'] = Features.MagFeature
#search_to_drop = AtomicFeaturesMagpie.columns.str.contains('range|min|max')
#columns_to_drop = AtomicFeaturesMagpie.loc[:,search_to_drop].columns
#AtomicFeaturesMagpie.drop(columns=columns_to_drop, inplace=True)
AtomicFeatures = pd.concat([AtomicFeaturesMagpie])
AtomicFeaturesMagpie.dropna(inplace=True)

In [13]:
AtomicFeatures

,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber,MagConfig
index,,,,,,,,,,,,,,,,,,,,,
Co_pv6W_sv6.C14-BBA.FM,27.0,74.0,47.0,69.727273,7.768595,74.0,51.0,58.0,7.0,51.636364,...,0.140770,0.255946,0.0,194.0,229.0,35.0,225.818182,5.785124,229.0,2
Co_pv6W_sv6.C14-BBA.NM,27.0,74.0,47.0,69.727273,7.768595,74.0,51.0,58.0,7.0,51.636364,...,0.140770,0.255946,0.0,194.0,229.0,35.0,225.818182,5.785124,229.0,0
Cr_pv6W_sv2.D0_19-A3B.FM,24.0,74.0,50.0,62.461538,17.751479,74.0,49.0,51.0,2.0,50.538462,...,0.000000,0.000000,0.0,229.0,229.0,0.0,229.000000,0.000000,229.0,2
Cr_pv6W_sv2.D0_19-A3B.NM,24.0,74.0,50.0,62.461538,17.751479,74.0,49.0,51.0,2.0,50.538462,...,0.000000,0.000000,0.0,229.0,229.0,0.0,229.000000,0.000000,229.0,0
Cr_pv16Co_pv4W_sv10.sigma-CBAAC.FM,24.0,74.0,50.0,41.066667,21.955556,24.0,49.0,58.0,9.0,50.866667,...,0.206463,0.357869,0.0,194.0,229.0,35.0,224.333333,8.088889,229.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Cr_pv20Co_pv2W_sv8.sigma-BAACA.FM,24.0,74.0,50.0,37.533333,19.448889,24.0,49.0,58.0,9.0,50.133333,...,0.103231,0.192699,0.0,194.0,229.0,35.0,226.666667,4.355556,229.0,2
Cr_pv20Co_pv2W_sv8.sigma-BAACA.NM,24.0,74.0,50.0,37.533333,19.448889,24.0,49.0,58.0,9.0,50.133333,...,0.103231,0.192699,0.0,194.0,229.0,35.0,226.666667,4.355556,229.0,0
Co_pv13W_sv16.chi-ABAB.NM,27.0,74.0,47.0,70.468208,6.532794,74.0,51.0,58.0,7.0,51.526012,...,0.116359,0.215231,0.0,194.0,229.0,35.0,226.369942,4.864847,229.0,0


In [14]:
thissamples = BS['EF'].index.intersection(AtomicFeaturesMagpie.index)
X_train, X_test, Y_train, Y_test = train_test_split(AtomicFeaturesMagpie.loc[thissamples], BS[TARGET][thissamples])

In [15]:
Params={'feature_selector__k': np.arange(10), 'regressor__max_depth': np.arange(20,30)}
model_selector = Pipeline(
    [
        ('feature_selector', SelectKBest()),
#        ('feature_selector', SequentialFeatureSelector(RandomForestRegressor())),
        ('regressor', RandomForestRegressor())
        
    ]
)

In [16]:
seqselector = SequentialFeatureSelector(RandomForestRegressor, n_features_to_select=10, n_jobs=3)

In [17]:
cv = GridSearchCV(model_selector, param_grid=Params,verbose=5, n_jobs=3,scoring='neg_root_mean_squared_error')

In [18]:
cv.fit(X_train, Y_train)

Fitting 5 folds for each of 100 candidates, totalling 500 fits


GridSearchCV(estimator=Pipeline(steps=[('feature_selector', SelectKBest()),
                                       ('regressor', RandomForestRegressor())]),
             n_jobs=3,
             param_grid={'feature_selector__k': array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
                         'regressor__max_depth': array([20, 21, 22, 23, 24, 25, 26, 27, 28, 29])},
             scoring='neg_root_mean_squared_error', verbose=5)

In [19]:
cv.best_estimator_

Pipeline(steps=[('feature_selector', SelectKBest(k=9)),
                ('regressor', RandomForestRegressor(max_depth=21))])

In [20]:
AtomicFeatures.loc[:,cv.best_estimator_[0].get_support()].to_pickle('tables/BestAtomicFeatures.pkl')

In [21]:
AtomicFeatures.loc[:,cv.best_estimator_[0].get_support()].columns

Index(['MagpieData avg_dev NdUnfilled', 'MagpieData range NUnfilled',
       'MagpieData avg_dev NUnfilled', 'MagpieData range GSvolume_pa',
       'MagpieData avg_dev GSvolume_pa', 'MagpieData range GSmagmom',
       'MagpieData avg_dev GSmagmom', 'MagpieData range SpaceGroupNumber',
       'MagpieData avg_dev SpaceGroupNumber'],
      dtype='object')

In [22]:
cv.best_score_

-0.13017523855144897

In [ ]:
seq

In [12]:
del FeatureConcatenate
FeatureConcatenate = SourceFileLoader('FeatureConcate','/home/storage/fortimtb/CuadernoTrabajo/bopfoxfeaturizer/BopFoxFeaturizer/FeatureConcatenate.py'
                                     ).load_module().FeatureConcatenate

param_grid = {} 
Bestfeatsatomic = {} 
Bestscoresatomic = {}
FC['RandomForest_atomic'] = FeatureConcatenate(
    pd.concat([AtomicFeaturesMagpie.loc[thissamples], BS[TARGET][thissamples]], axis=1), #DATA, 
    RandomForestRegressor(),
    model_params=param_grid,
    feature_list=AtomicFeaturesMagpie.columns,
    data_target=TARGET,
    criterion = criterion
#    sample_weights = w_train, #Classes['Weights']
)

Bestfeatsatomic['RandomForest_atomic'], Bestscoresatomic['RandomForest_atomic'] = FC['RandomForest_atomic'].build_features_list(
    best_feature_proposal=['MagConfig'],
    pass_force_refit=True,
    maxnumfeatures=15,
    report_prefix='RandomForest_atomic_'+CASE,
)

BestAtomic = pd.DataFrame(
    np.array([[x,y] for x,y in zip(Bestfeatsatomic['RandomForest_atomic'], Bestscoresatomic['RandomForest_atomic'])])
)

procesing '[]' with 'MagConfig' ... ::   0%|                                                      | 0/1 [00:00<?, ?it/s]

Refitting ..


procesing '[]' with 'MagConfig' ... :: 100%|##############################################| 1/1 [00:00<00:00,  1.38it/s]
procesing '['MagConfig']' with 'MagpieData mean NsValence' ... ::   0%|                          | 0/66 [00:00<?, ?it/s]

fitting has finished,  test_score  =  0.18839163757886096
Refitting ..


procesing '['MagConfig']' with 'MagpieData avg_dev NfUnfilled' ... :: 100%|#############| 66/66 [00:58<00:00,  1.13it/s]


...

fitting has finished,  test_score  =  0.12355251542344826


procesing '['MagConfig', 'MagpieData mean AtomicWeight', 'MagpieData avg_dev MeltingT', 'MagpieData mean NpValence', 'Ma

Refitting ..


procesing '['MagConfig', 'MagpieData mean AtomicWeight', 'MagpieData avg_dev MeltingT', 'MagpieData mean NpValence', 'Ma


fitting has finished,  test_score  =  0.12393955081646119


In [13]:
BestAtomic.to_csv(FileNames.get_table_filename('AtomicFeatures',''),index=None, header=None)

# Density Features

In [14]:
DensitiFeatures

,density,vpa,packing fraction
index,,,
Co_pv6W_sv6.C14-BBA.FM,19.558247,10.305995,1.0
Co_pv6W_sv6.C14-BBA.NM,19.558247,10.305995,1.0
Cr_pv6W_sv2.D0_19-A3B.FM,12.599284,11.197029,1.0
Cr_pv6W_sv2.D0_19-A3B.NM,12.599284,11.197029,1.0
Cr_pv16Co_pv4W_sv10.sigma-CBAAC.FM,14.703874,10.939619,1.0
...,...,...,...
Cr_pv20Co_pv2W_sv8.sigma-BAACA.FM,13.109662,11.098025,1.0
Cr_pv20Co_pv2W_sv8.sigma-BAACA.NM,13.109662,11.098025,1.0
Co_pv13W_sv16.chi-ABAB.NM,20.599218,10.305995,1.0


# Composition Features 

In [15]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

In [16]:
CompositionFeatures['MagConfig'] = Features.MagFeature

In [17]:
CompositionFeatures.dropna(axis=1,inplace=True)

In [18]:
text_columns = CompositionFeatures.select_dtypes(include=['object']).columns

In [19]:
float_columns = CompositionFeatures.select_dtypes(include=['float']).columns

In [20]:
from sklearn.pipeline import Pipeline

In [21]:
model = Pipeline(
    
)

TypeError: __init__() missing 1 required positional argument: 'steps'

In [ ]:
OHE = OneHotEncoder(sparse=True)

In [ ]:
OHE.fit_transform(CompositionFeatures[text_columns])

In [ ]:
del FeatureConcatenate
FeatureConcatenate = SourceFileLoader('FeatureConcate','/home/storage/fortimtb/CuadernoTrabajo/bopfoxfeaturizer/BopFoxFeaturizer/FeatureConcatenate.py'
                                     ).load_module().FeatureConcatenate

param_grid = {} 
Bestfeatsatomic = {} 
Bestscoresatomic = {}
FC['RandomForest_atomic'] = FeatureConcatenate(
    pd.concat([AtomicFeaturesMagpie.loc[thissamples], BS[TARGET][thissamples]], axis=1), #DATA, 
    RandomForestRegressor(),
    model_params=param_grid,
    feature_list=AtomicFeaturesMagpie.columns,
    data_target=TARGET,
    criterion = criterion
#    sample_weights = w_train, #Classes['Weights']
)

Bestfeatsatomic['RandomForest_atomic'], Bestscoresatomic['RandomForest_atomic'] = FC['RandomForest_atomic'].build_features_list(
    best_feature_proposal=['MagConfig'],
    pass_force_refit=True,
    report_prefix='RandomForest_atomic_'+CASE,
)

BestAtomic = pd.DataFrame(
    np.array([[x,y] for x,y in zip(Bestfeatsatomic['RandomForest_atomic'], Bestscoresatomic['RandomForest_atomic'])])
)

BestAtomic.to_csv(get_table_filename('AtomicFeatures',FC['RandomForest_atomic'].criterion))